In [24]:
import pandas as pd
import numpy as np  
import os
import matplotlib.pyplot as plt

BASE_DIR = os.path.dirname(os.getcwd())
DATA_PATH = os.path.join(BASE_DIR, 'data', "massive_financial_data.csv")
OUTPUT_PATH = os.path.join(BASE_DIR, 'data', "processed_data.csv")

print("Ruta del archivo de datos: ", DATA_PATH)

Ruta del archivo de datos:  c:\Users\Usuario\Desktop\bachelor-thesis\data\massive_financial_data.csv


### Celda de extracción del fichero de datos

In [25]:
# 1. Carga del CSV con MultiIndex (Tickers en L0, Métricas en L1)
raw_data = pd.read_csv(DATA_PATH, header=[0, 1], index_col=0, parse_dates=True)

# 2. Limpieza de Ingeniería: Forward Fill para evitar huecos por festivos o errores de red
raw_data = raw_data.ffill()

# 3. Extracción vectorizada de cada componente (fuel para nuestras métricas)
# Usamos .xs() indicando level=1 porque ahí residen 'Open', 'High', etc.
try:
    open_p = raw_data.xs('Open', axis=1, level=1)
    high_p = raw_data.xs('High', axis=1, level=1)
    low_p  = raw_data.xs('Low', axis=1, level=1)
    close  = raw_data.xs('Close', axis=1, level=1)
    volume = raw_data.xs('Volume', axis=1, level=1)
    
    print(f"Extracción completada.")
    print(f"Activos detectados (incluyendo Benchmark): {close.shape[1]}")
    print(f"Rango temporal: {close.index.min().date()} a {close.index.max().date()}")

    # Verificación de seguridad para el TFG: ¿Está el benchmark?
    if '^GSPC' in close.columns:
        print("Benchmark (^GSPC) detectado correctamente.")
    else:
        print("Alerta: No se encuentra ^GSPC en las columnas. Revisa el notebook 01.")

except KeyError as e:
    print(f"Error de extracción: {e}. Comprueba los nombres de las columnas en el CSV.")

C:\Users\Usuario\AppData\Local\Temp\ipykernel_20564\2401456902.py:2: DtypeWarning: Columns (6,12,18,24,30,36,42,48,54,60,66,72,78,84,90,96,102,108,114,120,126,132,138,144,150,156,162,168,174,180,186,192,198,204,210,216,222,228,234,240,246,252,258,264,270,276,282,288,294,300,306,312,318,324,330,336,342,348,354,360,366,372,378,384,390,396,402,408,414,420,426,432,438,444,450,456,462,468,474,480,486,492,498,504,510,516,522,528,534,540,546,552,558,564,570,576,582,588,594,600,606,612,618,624,630,636,642,648,654,660,666,672,678,684,690,696,702,708,714,720,726,732,738,744,750,756,762,768,774,780,786,792,798,804,810,816,822,828,834,840,846,852,858,864,870,876,882,888,894,900,906,912,918,924,930,936,942,948,954,960,966,972,978,984,990,996,1002,1008,1014,1020,1026,1032,1038,1044,1050,1056,1062,1068,1074,1080,1086,1092,1098,1104,1110,1116,1122,1128,1134,1140,1146,1152,1158,1164,1170,1176,1182,1188,1194,1200,1206,1212,1218,1224,1230,1236,1242,1248,1254,1260,1266,1272,1278,1284,1290,1296,1302,1308,1

Extracción completada.
Activos detectados (incluyendo Benchmark): 505
Rango temporal: 2005-01-03 a 2025-12-30
Benchmark (^GSPC) detectado correctamente.


C:\Users\Usuario\AppData\Local\Temp\ipykernel_20564\2401456902.py:5: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  raw_data = raw_data.ffill()


In [26]:
#Calculo de métricas financieras avanzadas (vectorizadas para eficiencia)

window = 20

# 1. Z-Score Clásico (Reversión a la media)
rolling_mean = close.rolling(window=window).mean()
rolling_std = close.rolling(window=window).std()
z_score = (close - rolling_mean) / rolling_std

# 2. Ratio de Amihud (Liquidez/Slippage)
returns_abs = close.pct_change().abs()
dollar_volume = close * volume
amihud = (returns_abs / (dollar_volume + 1e-15)).rolling(window=window).mean()

# 3. Volatilidad Garman-Klass (Estimador de varianza intradía)
# Ecuación matemática vectorizada
log_hl = np.log((high_p / (low_p + 1e-10)) + 1e-10)**2
log_co = np.log((close / (open_p + 1e-10)) + 1e-10)**2
gk_variance = 0.5 * log_hl - (2 * np.log(2) - 1) * log_co
gk_volatility = np.sqrt(gk_variance.rolling(window=window).mean() * 252)

# 4. RVOL (Relative Volume - Interés Institucional)
volume_ma = volume.rolling(window=window).mean()
rvol = volume / (volume_ma + 1e-10)

# 5. Momentum a 1 Mes (20 días hábiles)
momentum_1m = close.pct_change(periods=window)

In [27]:
# Calculo de la volatilidad ideosincrónica (RVOL) para el benchmark

returns = close.pct_change()

benchmark_returns = returns['^GSPC']

# Varianza rodante del mercado
var_market = benchmark_returns.rolling(window=window).var()

# Covarianza rodante de cada activo frente al mercado
# Al pasarle una Serie (benchmark) a un DataFrame, Pandas alinea todo mágicamente
cov_assets_market = returns.rolling(window=window).cov(benchmark_returns)

# Beta rodante (Sensibilidad al mercado)
rolling_beta = cov_assets_market.div(var_market, axis=0)

# Retorno Esperado en base al mercado (Beta * Retorno del Benchmark)
expected_returns = rolling_beta.multiply(benchmark_returns, axis=0)

# Residuos / Alpha (Lo que hace la acción por mérito propio)
residuals = returns - expected_returns

# Volatilidad Idiosincrásica (Desviación estándar de los residuos, anualizada)
idio_vol = residuals.rolling(window=window).std() * np.sqrt(252)


In [ ]:
# --- Consolidación del Feature Store ---
features = pd.DataFrame({
    'Z_Score': z_score.stack(future_stack=True),
    'Amihud_Ratio': amihud.stack(future_stack=True),
    'Garman_Klass_Vol': gk_volatility.stack(future_stack=True),
    'RVOL': rvol.stack(future_stack=True),
    'Momentum_1M': momentum_1m.stack(future_stack=True),
    'Idiosyncratic_Vol': idio_vol.stack(future_stack=True) # <- NUEVA
})

# IMPORTANTE: Eliminamos el benchmark (^GSPC) del dataset final.
features = features.drop('^GSPC', level=1)

# Limpiamos NaNs y nombramos índices
features = features.dropna()
features.index.names = ['Date', 'Ticker']

print(f"✅ Feature Store consolidado. Dimensiones: {features.shape}")

✅ Feature Store consolidado. Dimensiones: (2407474, 6)


In [30]:
print(features.head())

                    Z_Score  Amihud_Ratio  Garman_Klass_Vol      RVOL  \
Date       Ticker                                                       
2005-03-01 HIG     1.268989  1.703231e-10          0.148056  0.699351   
           AMT     1.557286  5.395887e-10          0.201258  0.993537   
           PWR    -0.936412  4.688760e-09          0.416153  1.798361   
           BAC     1.078270  3.371613e-11          0.109682  1.345941   
           MET     1.088575  1.958884e-10          0.183673  0.812221   

                   Momentum_1M  Idiosyncratic_Vol  
Date       Ticker                                  
2005-03-01 HIG        0.086441           0.146857  
           AMT        0.030353           0.222514  
           PWR        0.012032           0.325159  
           BAC        0.015311           0.081715  
           MET        0.033459           0.155256  


Cuando escribas la memoria, esta celda te permite justificar que has reducido la complejidad computacional de $O(N \times M \times T)$ a operaciones matriciales de bajo nivel, logrando procesar millones de datos en segundos. Eso justifica el grado de Ingeniería Informática al 100%.

In [29]:
# 1. Aseguramos que el directorio exista
output_dir = os.path.join(BASE_DIR, 'data')
os.makedirs(output_dir, exist_ok=True)

# 2. Ruta del archivo Parquet (más eficiente que CSV)
feature_store_path = os.path.join(output_dir, "feature_store.csv")

# 3. Guardado
features.to_csv(feature_store_path, index=False)

print(f"Feature Store guardado exitosamente en: {feature_store_path}")

Feature Store guardado exitosamente en: c:\Users\Usuario\Desktop\bachelor-thesis\data\feature_store.csv
